# Telco Network Logs - live simulator
## -> Microsoft Fabric Eventstream (Custom Endpoint)

Paste your Eventstream custom endpoint connection string in cell 2, then run every cell.
No files, no Lakehouse mount, no environment variables.

### Two log streams

| `event_type` | Grain | Contents |
|---|---|---|
| `cell.kpi` | one cell, every tick | PRB utilisation, RRC success, drop rate, throughput, users, RSRP/SINR |
| `network.alarm` | one event | Alarm raise / clear with severity, domain and affected cell |

Both carry keys that join to the Lakehouse CSVs: `cell_id`, `site_id`, `city`, `alarm_code`.

Every event also carries a **`payload` column holding a nested JSON object** with the detailed
counters for that event type. It lands in Eventhouse as a `dynamic` column, so you can query into
it with dot notation (`payload.radio.rsrp_dbm`) without pre-flattening the schema.

### Built-in incident

After 60 seconds, 10 cells in one city go into a **congestion incident** - PRB climbs above 90%,
throughput collapses, drop rate spikes, and matching alarms start firing. Events in the incident
are tagged `incident_id = "INC-001"` so one KQL filter shows the whole story.

The notebook **streams continuously until you stop it** - use the Fabric stop button or
*Interrupt kernel*. It prints a progress line every 10 seconds and a summary on exit.

## 1. Install the Event Hub SDK

In [ ]:
%pip install azure-eventhub --quiet

## 2. Connection

Fabric: **Eventstream -> your Custom endpoint source -> `Event hub` tab -> Connection string-primary key**.
Paste it below exactly as shown - it already ends with `;EntityPath=es_<id>`.

In [ ]:
CONNECTION_STRING = "Endpoint=sb://<namespace>.servicebus.windows.net/;SharedAccessKeyName=key_<id>;SharedAccessKey=<key>;EntityPath=es_<eventstream-id>"

EVENTS_PER_SECOND = 20    # streams until you stop the notebook

## 3. Build the network

Creates the cells the simulator will report on. IDs match `dim_cell.csv` / `dim_site.csv`
(`CELL-0001`..`CELL-0150`, `SITE-001`..`SITE-050`), so live events join straight to the Lakehouse
dimensions.

Each cell keeps a persistent `load` value, so readings drift realistically instead of jumping
around randomly between ticks.

In [ ]:
import random

random.seed(20261012)

CITIES = ["Sydney", "Melbourne", "Brisbane", "Perth", "Adelaide"]
BANDS = [("n78", "5G", 100), ("n28", "5G", 20),
         ("B3", "4G", 20), ("B28", "4G", 20), ("B7", "4G", 20)]

ALARMS = [
    ("ALM-001", "Cell Out of Service",       "Critical", "RAN"),
    ("ALM-002", "High PRB Utilisation",      "Major",    "RAN"),
    ("ALM-003", "RRC Setup Failure High",    "Major",    "RAN"),
    ("ALM-004", "Handover Failure High",     "Minor",    "RAN"),
    ("ALM-005", "RF Unit Temperature High",  "Major",    "RAN"),
    ("ALM-006", "Antenna / VSWR Fault",      "Major",    "RAN"),
    ("ALM-007", "Site Power Failure",        "Critical", "Power"),
    ("ALM-008", "Battery Low",               "Major",    "Power"),
    ("ALM-009", "Backhaul Link Down",        "Critical", "Transport"),
    ("ALM-010", "Backhaul Congestion",       "Major",    "Transport"),
    ("ALM-011", "High Packet Loss",          "Major",    "Transport"),
    ("ALM-012", "Core Session Failure High", "Major",    "Core"),
]

# 50 sites x 3 sectors = 150 cells, matching dim_cell.csv
network = []
cell_no = 0
for site_no in range(1, 51):
    city = random.choice(CITIES)
    for sector in (1, 2, 3):
        cell_no += 1
        band, tech, bw = random.choice(BANDS)
        network.append({
            "cell_id": f"CELL-{cell_no:04d}",
            "site_id": f"SITE-{site_no:03d}",
            "city": city,
            "technology": tech,
            "band": band,
            "max_throughput_mbps": round(bw * (7.5 if tech == "5G" else 5.0), 1),
            "load": random.uniform(0.2, 0.6),      # persistent, drifts over time
        })

# Incident: 10 cells in one city get congested after 60s
INCIDENT_CITY = "Sydney"
INCIDENT_CELLS = {c["cell_id"] for c in
                  random.sample([c for c in network if c["city"] == INCIDENT_CITY] or network, 10)}
INCIDENT_START = 60

print(f"{len(network)} cells across {len({c['site_id'] for c in network})} sites")
print(f"Incident cells ({INCIDENT_CITY}): {sorted(INCIDENT_CELLS)}")

## 4. Event generators

`cell_kpi()` produces a performance sample; `alarm()` produces a fault event. Both take the
elapsed run time so they know whether the incident is active.

Each event has a **`payload`** field containing a nested JSON object with the detailed counters -
this arrives in Eventhouse as a `dynamic` column you can query with dot notation.

In [ ]:
import json
import uuid
from datetime import datetime, timezone


def now():
    return datetime.now(timezone.utc).strftime("%Y-%m-%dT%H:%M:%S.%f")[:-3] + "Z"


def cell_kpi(elapsed):
    c = random.choice(network)
    hit = c["cell_id"] in INCIDENT_CELLS and elapsed > INCIDENT_START

    # drift the persistent load a little each tick
    c["load"] = min(0.98, max(0.05, c["load"] + random.gauss(0, 0.02)))
    load = min(0.99, c["load"] + 0.4) if hit else c["load"]

    prb = round(load * 100, 2)
    throughput = c["max_throughput_mbps"] * max(0.05, 1 - load * 0.85) * random.uniform(0.8, 1.1)
    rrc = 99.5 - max(0, prb - 75) * 0.35 + random.gauss(0, 0.3)
    drop = 0.3 + max(0, prb - 80) * 0.25 + random.gauss(0, 0.15)
    users = int(max(1, random.gauss(prb * 3, 20)))
    rsrp = round(random.gauss(-95 - prb * 0.1, 5), 1)
    sinr = round(random.gauss(14 - prb * 0.08, 3), 2)

    return {
        "event_type": "cell.kpi",
        "event_time": now(),
        "cell_id": c["cell_id"],
        "site_id": c["site_id"],
        "city": c["city"],
        "technology": c["technology"],
        "band": c["band"],
        "prb_utilisation_pct": prb,
        "rrc_success_rate_pct": round(max(80.0, min(99.9, rrc)), 2),
        "drop_rate_pct": round(max(0.02, drop), 3),
        "throughput_mbps": round(max(0.3, throughput), 2),
        "active_users": users,
        "rsrp_dbm": rsrp,
        "sinr_db": sinr,
        "congested": prb > 85,
        "incident_id": "INC-001" if hit else None,
        # nested JSON detail -> dynamic column in Eventhouse
        "payload": {
            "radio": {
                "rsrp_dbm": rsrp,
                "rsrq_db": round(random.gauss(-10.5, 1.5), 2),
                "sinr_db": sinr,
                "cqi": round(max(1.0, min(15.0, random.gauss(10 - prb * 0.05, 1.2))), 1),
            },
            "traffic": {
                "dl_volume_mb": round(max(0.5, throughput * 7.5 * random.uniform(0.6, 1.2)), 2),
                "ul_volume_mb": round(max(0.1, throughput * 1.6 * random.uniform(0.5, 1.1)), 2),
                "active_bearers": int(users * random.uniform(1.1, 1.8)),
                "handover_attempts": int(max(0, random.gauss(users * 0.3, 8))),
            },
            "resources": {
                "prb_used": int(prb * 2.73),
                "prb_available": 273 if c["technology"] == "5G" else 100,
                "cpu_load_pct": round(min(99.0, max(5.0, random.gauss(35 + prb * 0.4, 8))), 1),
                "temperature_c": round(random.gauss(38 + prb * 0.08, 4), 1),
            },
            "health": {
                "state": "Congested" if prb > 85 else "Normal",
                "alarm_active": hit,
                "uptime_hours": round(random.uniform(24, 8760), 1),
            },
        },
    }


def alarm(elapsed):
    incident_on = elapsed > INCIDENT_START

    # during the incident, mostly congestion alarms on the affected cells
    if incident_on and random.random() < 0.6:
        cell_id = random.choice(sorted(INCIDENT_CELLS))
        c = next(x for x in network if x["cell_id"] == cell_id)
        code_, name, severity, domain = random.choice(
            [a for a in ALARMS if a[0] in ("ALM-002", "ALM-003", "ALM-010", "ALM-012")])
        tagged = True
    else:
        c = random.choice(network)
        code_, name, severity, domain = random.choice(ALARMS)
        tagged = False

    state = random.choices(["raised", "cleared"], weights=[75, 25])[0]

    return {
        "event_type": "network.alarm",
        "event_time": now(),
        "alarm_event_id": "ALMEV-" + uuid.uuid4().hex[:10].upper(),
        "alarm_code": code_,
        "alarm_name": name,
        "severity": severity,
        "domain": domain,
        "cell_id": c["cell_id"] if domain == "RAN" else None,
        "site_id": c["site_id"],
        "city": c["city"],
        "state": state,
        "service_affecting": severity == "Critical",
        "incident_id": "INC-001" if tagged else None,
        # nested JSON detail -> dynamic column in Eventhouse
        "payload": {
            "source": {
                "managed_object": f"Site={c['site_id']},Cell={c['cell_id']}",
                "vendor": random.choice(["Ericsson", "Nokia", "Samsung"]),
                "technology": c["technology"],
                "band": c["band"],
            },
            "detail": {
                "probable_cause": random.choice(["Equipment Failure", "Congestion",
                                                 "Link Failure", "Power Problem",
                                                 "Threshold Crossed", "Degraded Signal"]),
                "threshold_value": round(random.uniform(70, 95), 1),
                "measured_value": round(random.uniform(85, 100), 1),
                "occurrence_count": random.randint(1, 25),
            },
            "handling": {
                "acknowledged": random.choice([True, False]),
                "auto_clearing": code_ in ("ALM-002", "ALM-003", "ALM-010", "ALM-012"),
                "field_dispatch": severity == "Critical" and random.random() < 0.5,
                "runbook": f"https://runbooks.telco.demo/{code_.lower()}",
            },
        },
    }


print("Sample cell.kpi event:")
print(json.dumps(cell_kpi(0), indent=2))

## 5. Stream to Eventstream

Runs **continuously until you stop it** — hit the stop button or *Interrupt kernel* when you've
seen enough. Progress prints every 10 seconds; the final counts print on exit.

Each event gets an `event_type` application property so you can split the stream with a Filter
transformation in Eventstream without parsing the body.

In [ ]:
import time
from collections import Counter
from azure.eventhub import EventHubProducerClient, EventData

producer = EventHubProducerClient.from_connection_string(CONNECTION_STRING)

counts = Counter()
start = time.time()
buffer = []
last_report = 0.0


def flush(buf):
    """Send everything in the buffer as one batch."""
    if not buf:
        return
    batch = producer.create_batch()
    for e in buf:
        data = EventData(json.dumps(e))
        data.properties = {"event_type": e["event_type"]}
        batch.add(data)
    producer.send_batch(batch)
    buf.clear()


print(f"Streaming at ~{EVENTS_PER_SECOND} events/sec. Stop the notebook to finish.\n")

try:
    while True:
        elapsed = time.time() - start

        # ~85% KPI samples, ~15% alarms
        event = cell_kpi(elapsed) if random.random() < 0.85 else alarm(elapsed)
        buffer.append(event)
        counts[event["event_type"]] += 1
        if event.get("incident_id"):
            counts["incident_tagged"] += 1

        if len(buffer) >= 50:
            flush(buffer)

        if elapsed - last_report >= 10:
            last_report = elapsed
            sent = counts["cell.kpi"] + counts["network.alarm"]
            mins, secs = divmod(int(elapsed), 60)
            print(f"{mins:02d}:{secs:02d} | sent {sent:7,} "
                  f"| kpi {counts['cell.kpi']:6,} alarm {counts['network.alarm']:5,} "
                  f"| incident {'ACTIVE' if elapsed > INCIDENT_START else 'not started'}")

        time.sleep(1.0 / EVENTS_PER_SECOND)

except KeyboardInterrupt:
    print("\nStopped.")

finally:
    flush(buffer)
    producer.close()
    total = counts["cell.kpi"] + counts["network.alarm"]
    runtime = int(time.time() - start)
    print(f"\nTotal sent: {total:,} events over {runtime // 60}m {runtime % 60}s")
    print(f"  cell.kpi        {counts['cell.kpi']:,}")
    print(f"  network.alarm   {counts['network.alarm']:,}")
    print(f"  incident-tagged {counts['incident_tagged']:,}")

## 6. Downstream in Fabric

### Eventstream routing
Add a **Filter** transformation on `event_type` and route to two Eventhouse tables:

| Filter | Table |
|---|---|
| `event_type == "cell.kpi"` | `CellKpi` |
| `event_type == "network.alarm"` | `NetworkAlarm` |

### Lakehouse
Upload the 6 CSVs from `data/` and load them to Delta tables. They join on `cell_id`, `site_id`,
`city` and `alarm_code`.

### KQL to get started

In [ ]:
kql = r"""
// 1. Congested cells right now
CellKpi
| where event_time > ago(10m)
| summarize prb = avg(prb_utilisation_pct),
            tput = avg(throughput_mbps),
            drop = avg(drop_rate_pct),
            users = max(active_users)
  by cell_id, site_id, city, technology, band
| where prb > 85
| order by prb desc

// 2. The incident - KPI collapse over time
CellKpi
| where incident_id == "INC-001"
| summarize avg_prb = avg(prb_utilisation_pct),
            avg_tput = avg(throughput_mbps),
            avg_drop = avg(drop_rate_pct)
  by bin(event_time, 30s)
| render timechart

// 3. Alarms by severity and city
NetworkAlarm
| where event_time > ago(1h) and state == "raised"
| summarize alarms = count(), cells = dcount(cell_id)
  by severity, domain, city
| order by alarms desc

// 4. Alarm bursts - 5+ alarms on one site in 5 minutes
NetworkAlarm
| where event_time > ago(1h) and state == "raised"
| summarize alarm_count = count(), codes = make_set(alarm_code)
  by site_id, bin(event_time, 5m)
| where alarm_count >= 5
| order by alarm_count desc

// 5. Live KPIs joined to the Lakehouse site dimension
CellKpi
| where event_time > ago(15m)
| summarize prb = avg(prb_utilisation_pct), tput = avg(throughput_mbps)
  by cell_id, site_id
| lookup kind=leftouter dim_site on site_id
| project site_name, city, morphology, vendor, cell_id, prb, tput
| order by prb desc

// 6. Which subscribers are on congested cells
CellKpi
| where event_time > ago(15m)
| summarize prb = avg(prb_utilisation_pct) by site_id
| where prb > 85
| join kind=inner (dim_subscriber) on $left.site_id == $right.home_site_id
| summarize affected = count(), vips = countif(vip_flag == "Y") by city, plan_name
| order by vips desc

// 7. Query into the nested JSON payload (dynamic column, dot notation)
CellKpi
| where event_time > ago(15m)
| extend cqi         = toreal(payload.radio.cqi),
         dl_volume   = toreal(payload.traffic.dl_volume_mb),
         prb_used    = toint(payload.resources.prb_used),
         cpu_load    = toreal(payload.resources.cpu_load_pct),
         temperature = toreal(payload.resources.temperature_c),
         health      = tostring(payload.health.state)
| summarize avg_cqi = avg(cqi), total_dl_gb = sum(dl_volume) / 1024,
            avg_cpu = avg(cpu_load), max_temp = max(temperature),
            congested_samples = countif(health == "Congested")
  by cell_id, city
| where avg_cpu > 70 or max_temp > 55
| order by avg_cpu desc

// 8. Alarm payload detail - threshold breaches needing field dispatch
NetworkAlarm
| where event_time > ago(1h) and state == "raised"
| extend cause       = tostring(payload.detail.probable_cause),
         threshold   = toreal(payload.detail.threshold_value),
         measured    = toreal(payload.detail.measured_value),
         occurrences = toint(payload.detail.occurrence_count),
         dispatch    = tobool(payload.handling.field_dispatch),
         vendor      = tostring(payload.source.vendor),
         runbook     = tostring(payload.handling.runbook)
| where dispatch == true
| extend breach_margin = measured - threshold
| project event_time, site_id, cell_id, city, alarm_name, severity,
          cause, vendor, threshold, measured, breach_margin, occurrences, runbook
| order by breach_margin desc
"""
print(kql)